In [3]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import fetch_olivetti_faces
from sklearn.preprocessing import OneHotEncoder
import logging

# Configure logging to overwrite "_.log" every run (mode="w")
logging.basicConfig(
    level=logging.INFO,
    format="%(message)s",
    handlers=[
        logging.FileHandler("q3b.log", mode="w"),  # "w" overwrites, "a" appends
        logging.StreamHandler()
    ],
    force=True  # Resets any existing logging handlers
)
logger = logging.getLogger(__name__)

faces = fetch_olivetti_faces(shuffle=False)
X = faces.images
y = faces.target

height = X.shape[1]
width = X.shape[2]

split_data = np.load("olivetti_split_indices.npz")

train_idx = split_data["train"]
val_idx = split_data["val"]
test_idx = split_data["test"]

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

X_test_corrupted = X_test.copy()
X_test_corrupted[:, 24:40, 24:40] = 0.0

logger.info(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
logger.info(f"X_val:   {X_val.shape}, y_val:   {y_val.shape}")
logger.info(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")
logger.info(f"X_test_corrupted:  {X_test_corrupted.shape}, y_test:  {y_test.shape}")




X_train = X_train.reshape(-1, height * width)
X_val = X_val.reshape(-1, height * width)
X_test = X_test.reshape(-1, height * width)
X_test_corrupted = X_test_corrupted.reshape(-1, height * width)

N = len(y_test)
R = 30
B = 100
classes = np.unique(y_train)
K = len(classes)
seed = 260007

encoder = OneHotEncoder(sparse_output=False, categories=[np.arange(K)])
e_targets = encoder.fit_transform(y_test.reshape(-1, 1))

def generate_resampled_indices(y_train, random_state):
    rng = np.random.default_rng(random_state)
    resampled_indices = []
    for cls_l in np.unique(y_train):
        cls_idx = np.where(y_train == cls_l)[0]
        chosen = rng.choice(cls_idx, size=len(cls_idx), replace=True)
        resampled_indices.extend(chosen)
    return np.array(resampled_indices)

def get_tree_proba(tree, X_flat, K):
    probas = tree.predict_proba(X_flat)
    if probas.shape[1] < K:
        full_p = np.zeros((X_flat.shape[0], K))
        full_p[:, tree.classes_] = probas
        return full_p
    return probas

def compute_bias_and_variance(probs_r, e):
    p_bar = np.mean(probs_r, axis=0)
    bias = np.mean(np.sum((p_bar - e) ** 2, axis=1))
    diff = probs_r - p_bar[np.newaxis, :, :]
    variance = np.mean(np.sum(diff ** 2, axis=2))
    return bias, variance

depths = [4, 16, None]
results = {}

for d in depths:
    depth_str = f"Depth={d}" if d is not None else "Depth=None"
    logger.info(f"--- Evaluating {depth_str} ---")

    st_probs = np.zeros((R, N, K))
    bag_probs = np.zeros((R, N, K))

    st_accs = []
    bag_accs = []

    for r in range(R):
        resample_idx = generate_resampled_indices(y_train, seed + r)
        X_tr_r, y_tr_r = X_train[resample_idx], y_train[resample_idx]

        st = DecisionTreeClassifier(max_depth=d, min_samples_leaf=2, random_state=seed + r)
        st.fit(X_tr_r, y_tr_r)
        p_st = get_tree_proba(st, X_test, K)
        st_probs[r] = p_st
        st_accs.append(np.mean(np.argmax(p_st, axis=1) == y_test))

        # B. 100-Tree Bagged Ensemble
        bag_trees = []
        b_rng = np.random.default_rng(seed + r)
        for b in range(B):
            boot_idx = b_rng.choice(len(y_tr_r), size=len(y_tr_r), replace=True)
            clf = DecisionTreeClassifier(max_depth=d, min_samples_leaf=2, random_state=seed + b)
            clf.fit(X_tr_r[boot_idx], y_tr_r[boot_idx])
            bag_trees.append(clf)

        all_p = np.array([get_tree_proba(clf, X_test, K) for clf in bag_trees])
        p_bag = np.mean(all_p, axis=0)
        bag_probs[r] = p_bag
        bag_accs.append(np.mean(np.argmax(p_bag, axis=1) == y_test))

    b_st, var_st = compute_bias_and_variance(st_probs, e_targets)
    b_bag, var_bag = compute_bias_and_variance(bag_probs, e_targets)
    
    var_reduction = ((var_st - var_bag) / var_st) * 100

    # results[d] = {
    #     "st_bias": b_st, "st_var": var_st, "st_acc": np.mean(st_accs), "st_std": np.std(st_accs),
    #     "bag_bias": b_bag, "bag_var": var_bag, "bag_acc": np.mean(bag_accs), "bag_std": np.std(bag_accs),
    #     "var_reduction": var_reduction
    # }
    logger.info(f"Depth:{d} | Model:{'Single Tree'} | Bias:{b_st:.4f}  | Variance:{var_st:.4f}   | Mean Acc:{np.mean(st_accs):.4f}   | N/A")
    logger.info(f"{'         '} | Model:{'100-Bagged'} | Bias:{b_bag:.4f}  | Variance:{var_bag:.4f}   | Mean Acc:{np.mean(bag_accs):.4f}   | Var Reduction (%):{var_reduction:.2f}%")
    logger.info("-" * 90)

# logger.info("\n" + "=" * 90)
# logger.info(f"{'Depth':<12} | {'Model':<12} | {'Bias':<8} | {'Variance':<8} | {'Mean Acc':<9} | {'Var Reduction (%)':<18}")
# logger.info("=" * 90)

# for d in depths:
#     d_label = str(d) if d is not None else "None"
#     res = results[d]
#     logger.info(f"{d_label} | {'Single Tree'} | {res['st_bias']:.4f}  | {res['st_var']:.4f}   | {res['st_acc']:.4f}   | N/A")
#     logger.info(f"{'   '} | {'100-Bagged'} | {res['bag_bias']:.4f}  | {res['bag_var']:.4f}   | {res['bag_acc']:.4f}   | {res['var_reduction']:.2f}%")
#     logger.info("-" * 90)

X_train: (240, 64, 64), y_train: (240,)
X_val:   (80, 64, 64), y_val:   (80,)
X_test:  (80, 64, 64), y_test:  (80,)
X_test_corrupted:  (80, 64, 64), y_test:  (80,)
--- Evaluating Depth=4 ---


KeyboardInterrupt: 